In [1]:
# config file read
import configparser
import pandas as pd
import numpy as np
import os

# read write files
#import xarray as xr


## Data load

### inspect example data

The format of the data file should ideally be:

        FREQ f1,f2,f3,...

        XPSN x1,x2,x3,...
        YPSN y1,y2,y3,...
        ZPSN z1,z2,z3,...


        CO,QUAD

        Re111,Im111
        Re112,Im112
        Re113,Im113,
        ...
    
        Re121,Im121,
        Re122,Im122,
        ...
        Re211,Im211,
        ...

Where the frequency vector (FREQ) is in Hz, the Y (YPSN) and Z(ZPSN) are optional, and Re/ImSPQ means the real/imaginary component of Gamma (which has axes freq, sensor, sensor) for frequency S between sensors P and Q. The read-in Gamma should be something like CO.reshape(NFREQ,NREC,NREC) + 1j * QUAD.reshape(NFREQ,NREC,NREC where NFREQ=len(FREQ) and NREC=len(XPSN).

If anything is unclear don't hesitate to ask/pop down.

In [2]:
data_path = "R:\Panacousticon_Code\eg_data.npz"
eg_data = np.load(data_path)

# print each variable name (key) in eg_data
print("Variables in eg_data:")
for key in eg_data.keys():
    print(f"    {key}")

# extract variables from eg_data
freq = eg_data['f'] ; print(f" f shape: {freq.shape}") # Hz, (N_freq,)
x    = eg_data['x'] ; print(f" x shape: {x.shape}") # m (N_receiver, dim)
Gamma = eg_data['Gamma'] ; print(f" Gamma shape: {Gamma.shape}") # real (N_freq, N_receiver, N_receiver)


if np.allclose(Gamma, np.conj(np.swapaxes(Gamma, -1, -2))):
    print(">>> CHECKED: Gamma is Hermitian.")
if not np.allclose(Gamma, np.conj(np.swapaxes(Gamma, -1, -2))):
    raise ValueError(f"Gamma is not Hermitian.")


Variables in eg_data:
    f
    x
    Gamma
 f shape: (3,)
 x shape: (3, 2)
 Gamma shape: (3, 3, 3)
>>> CHECKED: Gamma is Hermitian.


### test functions

In [ ]:
# FUNC rows of gamma
def row(values):
    """
    Function to convert a 1D array of values to a comma-separated string with 17 significant digits.

    """
    return ",".join(f"{v:.17g}" for v in values) # 17 digits: exact float64 round trip

# FUNC convert arrays to txt input file 
def convert_to_txt(frequency, x, Gamma, output_path, n_obs, out_order='C'):
    """
    Convert the frequency, x, and Gamma arrays to text files.

    Input
    -----
    frequency (np.ndarray): Frequency array [Hz]. (N_freq,)
    x (np.ndarray): Receiver positions array. (N_rec, dims)
    Gamma (np.ndarray): Transfer function array. (N_freq, N_rec, N_rec)
    output_path (str): Path to the output text file.
    n_obs (int): Number of observations (N_obs).
    out_order (str): Order in which Gamma is flattened. Default is 'C'
        (receiver q varies fastest, then receiver p, then frequency).
    
    Output
    ------
    file in txt format that contains: 
         freq
         FREQ f1,f2,...
         X
        (Y)
        (Z)
         XPSN x1,x2,...
        (YPSN y1,y2,...)
        (ZPSN z1,z2,...)
         Co, Quad
         re111, im111,
         re112, im112,
         CO,QUAD
         re111, im111
         re112, im112
         ...

    """

    # out_order 
    if out_order not in ['C', 'F']:
        raise ValueError("out_order must be 'C' or 'F'.")
   

    # Get the dimensions of the input arrays
    N_freq = frequency.shape[0] # number of frequency points
    N_rec = x.shape[0] # number of receiver positions
    N_dim = x.shape[1] # number of coordinate dimensions
   
    # make sure the frequency, x, and Gamma arrays have compatible shapes
    if frequency.ndim != 1:
        raise ValueError("Frequency array must be 1D.")

    # check that Gamma has 3 dims 
    if Gamma.ndim != 3:
        raise ValueError("Gamma array must be 3D.")

    # check that Gamma.shape is N_freq x N_rec x N_rec
    if Gamma.shape != (N_freq, N_rec, N_rec):
        raise ValueError(f"Gamma shape {Gamma.shape} does not match expected shape {(N_freq, N_rec, N_rec)}.")

    if N_dim > 3:
        raise ValueError("x can have at most 3 dimensions (XPSN, YPSN, ZPSN).")


    # tests done, print data info 
    print(f"Data info:")
    print(f">>> Number of frequency points (N_freq): {N_freq}")
    print(f">>> Number of receiver positions (N_rec): {N_rec}")
    print(f">>> Number of coordinate dimensions (N_dim): {N_dim}")
    print(f">>> Number of observations (N_obs): {n_obs}")

    
    # build the output file
    output_file = output_path + "_topophone2.txt"
     
    # 'C' order gives (freq, p, q) with q fastest: Re111, Re112, ..., Re121, ..., Re211, ...
    # so it is read back with CO.reshape(N_freq, N_rec, N_rec)
    if out_order == 'C':
        print("\nFlattening Gamma in 'C' order (receiver q varies fastest, then receiver p, then frequency).")
    elif out_order == 'F':
            print("\nFlattening Gamma in 'F' order (frequency varies fastest, then receiver p, then receiver q).")

    Gamma_flat = Gamma.reshape(-1, order=out_order)
    co_quad = np.column_stack([Gamma_flat.real, Gamma_flat.imag])
    with open(output_file, "w") as f:

        # write the order of the flattened Gamma
        f.write(f"ORDER {out_order}\n\n")

        # write the number of observations
        f.write(f"NOBS {n_obs}\n\n")

        # write the frequency and receiver positions
        f.write(f"FREQ {row(frequency)}\n\n")

        # write the receiver positions
        for label, coords in zip(["XPSN", "YPSN", "ZPSN"], x.T):
            f.write(f"{label} {row(coords)}\n")

        # write the CO, QUAD data 
        f.write("\n\nCO,QUAD\n\n")
        np.savetxt(f, co_quad, delimiter=",", fmt="%.17g")

 
    print(f"\nData successfully written to: {output_file}")
    
# FUNC read txt input 
def read_txt(input_file, order='C'):
    """
    Read the frequency, x, and Gamma arrays from a text file.

    Input
    -----
    input_file (str): Path to the input text file.
    order (str): Order in which Gamma is flattened. Default is 'C'
        (receiver q varies fastest, then receiver p, then frequency). 

    Output
    ------
    frequency (np.ndarray): Frequency array. (N_freq,)
    x (np.ndarray): Receiver positions array. (N_rec, dims)
    Gamma (np.ndarray): Transfer function array. (N_freq, N_rec, N_rec)
    """

    print(f"\nReading data from: {input_file} ............................\n")
    with open(input_file, "r") as f:
        lines = f.readlines()

    # Read order
    # check if ORDER line exists, if not, use the provided order
    if any(line.startswith("ORDER") for line in lines):

        order_line = next(line for line in lines if line.startswith("ORDER"))
        order = order_line.split()[1].strip()
        
        if order not in ['C', 'F']:
            raise ValueError(f"Invalid order '{order}' found in {input_file}. Must be 'C' or 'F'.")

        print(f"ORDER line found in input file. Using order '{order}':")
        if order == 'C':
            print(">>> Reading Gamma in 'C' order (receiver q varies fastest, then receiver p, then frequency).\n")
        if order == 'F':
            print(">>> Reading Gamma in 'F' order (frequency varies fastest, then receiver p, then receiver q).\n") 

    else:
        print(f"WARNING: ORDER line NOT found in input file. Using provided order '{order}'.")

        if order not in ['C', 'F']:
                    raise ValueError(f"Invalid order '{order}' found in {input_file}. Must be 'C' or 'F'.")
    
        if order == 'C':
            print(">>> Reading Gamma in 'C' order (receiver q varies fastest, then receiver p, then frequency).\n")
        if order == 'F':
            print(">>> Reading Gamma in 'F' order (frequency varies fastest, then receiver p, then receiver q).\n")

    # Read number of observations
    if not any(line.startswith("NOBS") for line in lines): # check if NOBS line exists, if not, raise an error
        raise ValueError(f"NOBS line not found in {input_file}. Please include the number of observations (NOBS) in the input file.")
    
    nobs_line = next(line for line in lines if line.startswith("NOBS"))
    n_obs = int(nobs_line.split()[1])

    # Read frequency
    freq_line = next(line for line in lines if line.startswith("FREQ"))
    frequency = np.array([float(val) for val in freq_line.split()[1].split(",")])

    # Read receiver positions
    x_lines = [line for line in lines if any(line.startswith(label) for label in ["XPSN", "YPSN", "ZPSN"])]
    x = np.array([[float(val) for val in line.split()[1].split(",")] for line in x_lines]).T

    # Read CO, QUAD data
    co_quad_start = next(i for i, line in enumerate(lines) if line.startswith("CO,QUAD")) + 1
    co_quad_data = np.loadtxt(lines[co_quad_start:], delimiter=",")
    
    # getting the dimensions of the data
    N_freq = frequency.shape[0]  
    N_rec = x.shape[0]  
    N_dim = x.shape[1]

    print(f"Data info:")
    print(f">>> Number of frequencies (N_freq): {N_freq}")
    print(f">>> Number of receivers (N_rec): {N_rec}")
    if N_rec < 2:
        raise ValueError("At least 2 receivers are required.")
    print(f">>> Number of coordinate dimensions (N_dim): {N_dim}")
    print(f">>> Number of observations (N_obs): {n_obs}")

    # check that the number of rows in co_quad_data matches N_freq * N_rec * N_rec   
    if co_quad_data.shape[0] != N_freq * N_rec * N_rec:
        raise ValueError("Mismatch between CO, QUAD data and expected shape based on frequency and receiver positions.")

    # reshape co_quad_data into a complex array and then into Gamma
    Gamma_flat = co_quad_data[:, 0] + 1j * co_quad_data[:, 1]

    # reshape Gamma_flat into (N_freq, N_rec, N_rec) using the specified order
    Gamma = Gamma_flat.reshape((N_freq, N_rec, N_rec), order=order)

    # check that Gamma is Hermitian
    if np.allclose(Gamma, np.conj(np.swapaxes(Gamma, -1, -2)), atol=1e-8 * np.abs(Gamma).max()):
        print("\nCHECKED: Gamma is Hermitian.")
    else:
        raise ValueError("WARNING:Gamma is not Hermitian.")

    print(f"\nData successfully read from: {input_file}.\n")

    return frequency, x, Gamma, n_obs

In [23]:
convert_to_txt(frequency=freq, x=x, Gamma=Gamma, n_obs = 200, output_path="R:\Panacousticon_Code\input_exampledata")

Data info:
>>> Number of frequency points (N_freq): 3
>>> Number of receiver positions (N_rec): 3
>>> Number of coordinate dimensions (N_dim): 2
>>> Number of observations (N_obs): 200

Flattening Gamma in 'C' order (receiver q varies fastest, then receiver p, then frequency).

Data successfully written to: R:\Panacousticon_Code\input_exampledata_topophone2.txt


In [47]:
read_freq, read_x, read_Gamma, read_n_obs = read_txt("R:\Panacousticon_Code\input_exampledata_topophone2.txt")


Reading data from: R:\Panacousticon_Code\input_exampledata_topophone2.txt ............................

ORDER line found in input file. Using order 'C':
>>> Reading Gamma in 'C' order (receiver q varies fastest, then receiver p, then frequency).

Data info:
>>> Number of frequencies (N_freq): 3
>>> Number of receivers (N_rec): 3
>>> Number of coordinate dimensions (N_dim): 2
>>> Number of observations (N_obs): 200

CHECKED: Gamma is Hermitian.

Data successfully read from: R:\Panacousticon_Code\input_exampledata_topophone2.txt.



In [43]:
read_x.shape
read_freq

array([3.96850263, 5.        , 6.29960525])

In [44]:
read_Gamma.shape

(3, 3, 3)

## Tests

In [2]:
config = configparser.ConfigParser()
config.read('config.ini')

['config.ini']

In [ ]:
# list of strings
signals = config["array_covariance_model"]["signals"].replace(" ", "").split(",")
print(f"Signals: {signals}")


Signals: ['normal', 'normal', 'point']


In [47]:
# make function that reads csv files and returns an array 
def read_csv_to_array(file_path):
    df = pd.read_csv(file_path, header=None)
    return df.values

a = read_csv_to_array('./data_1/x.csv')

print(f"Array shape: {a.shape}")
print(f"Array contents: {a}")

Array shape: (14, 3)
Array contents: [[1 2 3]
 [1 2 3]
 [1 2 3]
 [1 2 3]
 [1 2 3]
 [1 2 3]
 [1 2 3]
 [1 2 3]
 [1 2 3]
 [1 2 3]
 [1 2 3]
 [1 2 3]
 [1 2 3]
 [1 2 3]]


In [88]:
# floats, integers, bools
s = config["noise_model"].getfloat("s", fallback=None)
print(f"Noise model parameter s: {s}") 

n_obs = config["posterior_pdf"].getint("n_obs", fallback=None)
print(f"Number of observations: {n_obs}")

mean_field = config["inversion"].getboolean("mean_field", fallback=True)
print(f"Mean field: {mean_field}, type: {type(mean_field)}")

Noise model parameter s: 2.1
Number of observations: 300
Mean field: False, type: <class 'bool'>


In [105]:
# make arrays from list of strings in config file
def parse_array_from_config(string_list):
    return np.array([float(x) for x in string_list])

a = parse_array_from_config(config["noise_model"]["a"].replace(" ", "").split(","))
print(f"Sensor positions: {type(a)}, shape: {a.shape}")

Sensor positions: <class 'numpy.ndarray'>, shape: (2,)


## Making the parser function

In [10]:
def parse_array_from_config(config, section, option):
    entry = config.get(section, option, fallback=None)
    if entry is None:
        #print(f"The optional entry '{option}' not found in section '{section}' of the config file. It is set to None.")
        return None    
    else: 
        string_list = entry.replace(" ", "").split(",")
        return np.array([float(x) for x in string_list])

def read_csv_to_array(config, section, option):
    entry = config.get(section, option, fallback=None)
    if entry is None:
        #print(f"The optional entry '{option}' not found in section '{section}' of the config file. It is set to None.")
        return None
    else: 
        # check if the file exists
        file_path = entry
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File {file_path} does not exist.")
        
        df = pd.read_csv(file_path, header=None)

        # raise an error if the file is emplty
        if df.empty:
            raise ValueError(f"File {file_path} is empty.")
    
        return df.values

def sigs_input(signals, dim , signal_noise , a , b , s , max_order , sensor_selfnoise):
    sigs = []
    sigs.extend(signals)
    if sensor_selfnoise: sigs.append('selfnoise')

    noise_dict = {'sig':signal_noise}
    if (signal_noise == 'isonoise'):
        if dim:
            noise_dict['dim'] = dim
        else:
            raise ValueError("The 'dim' parameter must be provided in the config file for 'isonoise'.")
    if s: 
        noise_dict['s'] = s
    if (signal_noise in ['noise2D', 'noise3D']):
        if (a is not None) and (b is not None):
            if len(a) != len(b):
                raise ValueError("The lengths of 'a' and 'b' must be the same.")
            noise_dict['a'] = a
            noise_dict['b'] = b
        elif max_order: 
            noise_dict['max_order'] = max_order
        else:
            raise ValueError("Either 'a' and 'b' or 'max_order' must be provided in the config file.")

    sigs.append(noise_dict)

    return sigs


In [ ]:
def parse_config(path_to_config_file):
    """
    Convert the configuration file to variables.

    Input
    -----

    path_to_config_file : str

    Output
    ------
    inputs : dict
        Dictionary containing all the variables read from the configuration file.

    """

    # read the configuration file ===============================
    config = configparser.ConfigParser()
    config.read(path_to_config_file)

    # read input data from path specified in config file =======
    # signal data
    data_input = config.get("data", "signal_data_path") 
    
    if not os.path.exists(data_input):
        raise FileNotFoundError(f">> Signal data file {data_input} does not exist.")
    freq_Hz, sensor_positions, Gamma, n_obs = read_txt(data_input)

    # noise data
    noise_data_input = config.get("data", "noise_data_path", fallback=None)
    # if no input file is specified, set noise data to None
    if not noise_data_input:
        print("\nNo noise data file specified in config file. Setting Gamma_noise to None.\n")
        Gamma_noise = None
        n_obs_noise = None
    else:
        print(f"\nReading noise data from: {noise_data_input}\n")
        if not os.path.exists(noise_data_input):
            raise FileNotFoundError(f">> Noise data file {noise_data_input} does not exist.")
        _, _, Gamma_noise, n_obs_noise = read_txt(noise_data_input)

    # read variables from file ==================================   
    angular_freq = 2 * np.pi * freq_Hz
    signals = config["signal_model"]["signals"].replace(" ", "").split(",")
    weight_units = config["signal_model"].get("weight_units", fallback="dB")

    signal_noise = config["noise_model"]["signal_noise"].replace(" ", "").split(",")[0]
    dim = config["noise_model"].getint("dim", fallback=None)
    s = config["noise_model"].getfloat("s", fallback=None)
    a = parse_array_from_config(config, "noise_model", "a")
    b = parse_array_from_config(config, "noise_model", "b")
    max_order = config["noise_model"].getint("max_order", fallback=None)
    sensor_selfnoise = config["noise_model"].getboolean("sensor_selfnoise", fallback=False)

    bounds = read_csv_to_array(config, "posterior_pdf", "bounds")

    n_flows = config["inversion"].getint("n_flows", fallback=None)
    mean_field = config["inversion"].getboolean("mean_field", fallback=False)
    initial_distr = config["inversion"].get("initial_distr", fallback="Uniform")
    learning_rate = config["inversion"].getfloat("learning_rate", fallback=None)
    iterations = config["inversion"].getint("iterations", fallback=None)
    n_sample = config["inversion"].getint("n_sample", fallback=None)
    inversion_n_out = config["inversion"].getint("n_out", fallback=None)
    verbose = config["inversion"].getboolean("verbose", fallback=False)
    posterior_samples = config["inversion"].getint("posterior_samples", fallback=None)

    inversion_outpath = config.get("save_out", "inversion_outpath", fallback=None) 
    model_outpath = config.get("save_out", "model_outpath")
    loss_outpath = config.get("save_out", "loss_outpath", fallback=None)

    # compose other variables ==========================================

    save_intermediate_result = False
    if inversion_outpath is not None:
        save_intermediate_result = True 

    custom_cf = None 

    sigs = sigs_input(signals, dim , signal_noise , a , b , s , max_order , sensor_selfnoise)


    # make dict to hold variables ==========================
    args = dict(
                Gamma = Gamma,
                Gamma_noise = Gamma_noise,
                Nobs = n_obs,
                Nobs_n = n_obs_noise,
                x = sensor_positions,
                freq = angular_freq,
                sigs = sigs,
                weight_units = weight_units,
                custom_cf = custom_cf,
                prior_bounds = bounds,
                N_flow = n_flows,
                mean_field = mean_field,
                ini_dist = initial_distr,
                learning_rate = learning_rate,
                outpath = inversion_outpath,
                iterations = iterations,
                nsample = n_sample,
                nout = inversion_n_out,
                save_intermediate_result = save_intermediate_result,
                verbose = verbose,
                N_samples = posterior_samples,
                model_outpath = model_outpath,
                loss_outpath = loss_outpath
            )

    return args

In [51]:
args = parse_config('config.ini')


Reading data from: ./data_1/data_input.txt ............................

ORDER line found in input file. Using order 'C':
>>> Reading Gamma in 'C' order (receiver q varies fastest, then receiver p, then frequency).

Data info:
>>> Number of frequencies (N_freq): 3
>>> Number of receivers (N_rec): 3
>>> Number of coordinate dimensions (N_dim): 2
>>> Number of observations (N_obs): 200

CHECKED: Gamma is Hermitian.

Data successfully read from: ./data_1/data_input.txt.


No noise data file specified in config file. Setting Gamma_noise to None.



In [52]:
args

{'Gamma': array([[[ 1.00000000e+09+0.00000000e+00j,
           6.37048289e+08+1.43674707e+07j,
          -1.37775538e+08-8.62292939e+08j],
         [ 6.37048289e+08-1.43674707e+07j,
           1.00000000e+09+0.00000000e+00j,
          -1.61590966e+08-8.59270222e+08j],
         [-1.37775538e+08+8.62292939e+08j,
          -1.61590966e+08+8.59270222e+08j,
           1.00000000e+09+0.00000000e+00j]],
 
        [[ 1.00000000e+09+0.00000000e+00j,
           4.84763248e+08+1.03917874e+07j,
           5.62495313e+08-5.77745900e+08j],
         [ 4.84763248e+08-1.03917874e+07j,
           1.00000000e+09+0.00000000e+00j,
           5.43869087e+08-5.97507660e+08j],
         [ 5.62495313e+08+5.77745900e+08j,
           5.43869087e+08+5.97507660e+08j,
           1.00000000e+09+0.00000000e+00j]],
 
        [[ 1.00000000e+09+0.00000000e+00j,
           3.12426589e+08+1.82528476e+06j,
           2.14669698e+08-6.77293097e+08j],
         [ 3.12426589e+08-1.82528476e+06j,
           1.00000000e+09+0.0000

In [ ]:
def decompose(
    Gamma         : np.ndarray,
    Gamma_n       : np.ndarray = None,
    Nobs          : int,
    Nobs_n        : int        = None,
    x             : np.ndarray,
    freq          : np.ndarray,
    sigs          : list,
    weight_units  : str        = 'dB',
    custom_cf     : dict       = None,
    prior_bounds  : tuple,
    N_flow        : int,
    mean_field    : bool       = False,
    ini_dist      : str        = 'Uniform',
    learning_rate : float,
    outpath       : str        = '.',
    iterations    : int,
    nsample       : int,
    nout          : int        = 1,
    save_intermediate_result : bool = False,
    verbose       : bool       = False,
    N_samples     : int,
    model_outpath : str,
    loss_outpath  : str,
) -> np.ndarray:
    r'''
    `Gamma` : `numpy.ndarray`
        Observed array-covariance matrix
        (frequency, sensor, sensor)

    `Gamma_n` : `numpy.ndarray`
        Observed noise-only array-covariance matrix
        (frequency, sensor, sensor)

    `Nobs` : `int`
        Degrees of freedom

    `Nobs_n` : `int`
        Degrees of freedom (noise)

    `x` : `numpy.ndarray`
        Sensor position [m]
        (sensor, direction)

    `freq` : `numpy.ndarray`
        Angular frequency vector [rad.s-1]
        (frequency)

    `sigs` : `list`
        List of signals

    `weight_units` : `str`
        Units of the signal weights in the model vector
        ('dB' or 'linear')

    `custom_cf` : `dict`
        Custom characteristic functions keyed by signal name

    `prior_bounds` : `(numpy.ndarray,numpy.ndarray)`
        Prior bounds

    `N_flow` : `int`
        Number of normalising flows

    `mean_field` : `bool`
        Use mean-field approximation

    `ini_dist` : `str`
        Initial distribution to transform

    `learning_rate` : `float`

    `outpath` : `str`


    '''

    #__ Set up array-covariance model __
    if Gamma_n : calc_noise_only = True
    else:        calc_noise_only = False
    model = modelling.ArrayCovarianceModel(
        x               = x,
        freq            = freq,
        signal_types    = sigs,
        weight_units    = weight_units,
        calc_noise_only = calc_noise_only,
        custom_cf       = custom_cf
    )

    #__ Set up prior pdf __
    prior = inversion.Uniform(*prior_bounds)

    #__ Set up posterior pdf __
    posterior = inversion.Posterior(
        Gamma_obs             = Gamma,
        N_obs                 = Nobs,
        arraycovariance_model = model,
        prior                 = prior,
        Gamma_obs_noise       = Gamma_n,
        N_obs_noise           = Nobs_n
    )

    #__ Set up normalising flow model __
    flow = inversion.LinearFlowsModel(
        N_param      = len(prior_bounds[0]),
        prior_bounds = prior_bounds,
        N_flow       = N_flow,
        mean_field   = mean_field,
        ini_dist     = ini_dist
    )

    #__ Set up inverse problem __
    inverse = inversion.VI(
        flow_model    = flow,
        posterior     = posterior,
        learning_rate = learning_rate,
        outpath       = outpath
    )

    #__ Perform inversion __
    inverse.invert(
        iterations = iterations,
        nsample    = nsample,
        nout       = nout,
        save_intermediate_result = save_intermediate_result,
        verbose    = verbose
    )
    loss = inverse.loss_his

    #__ Sample posterior __
    m = inverse.sample(N_samples=N_samples)

    #__ Save models __
    if model_outpath: np.savetxt(model_outpath, m)
    if loss_outpath : np.savetxt(loss_outpath,  loss)

    # #__ Unpack __
    # if unpack: m = inverse.unpack(m)

    # #__ Return output __
    # if return_loss: return m, loss
    # else:           return m

## Test with package code

In [3]:
from topophone2.utils.read import parse_config

In [5]:
parse_config("/exports/csce/datastore/geos/users/s1803569/code/git-clones/topophone2/testing/config.ini")


Reading data from: /exports/csce/datastore/geos/users/s1803569/code/git-clones/topophone2/topophone2/data_1/data_input.txt ............................

ORDER line found in input file. Using order 'C':
>>> Reading Gamma_obs in 'C' order (receiver q varies fastest, then receiver p, then frequency).

Data info:
>>> Number of frequencies (N_freq): 3
>>> Number of receivers (N_rec): 3
>>> Number of coordinate dimensions (N_dim): 2
>>> Number of observations (N_obs): 200

CHECKED: Gamma_obs is Hermitian.

Data successfully read from: /exports/csce/datastore/geos/users/s1803569/code/git-clones/topophone2/topophone2/data_1/data_input.txt.


No noise data file specified in config file. Setting Gamma_obs_noise to None.



{'Gamma_obs': array([[[ 1.00000000e+09+0.00000000e+00j,
           6.37048289e+08+1.43674707e+07j,
          -1.37775538e+08-8.62292939e+08j],
         [ 6.37048289e+08-1.43674707e+07j,
           1.00000000e+09+0.00000000e+00j,
          -1.61590966e+08-8.59270222e+08j],
         [-1.37775538e+08+8.62292939e+08j,
          -1.61590966e+08+8.59270222e+08j,
           1.00000000e+09+0.00000000e+00j]],
 
        [[ 1.00000000e+09+0.00000000e+00j,
           4.84763248e+08+1.03917874e+07j,
           5.62495313e+08-5.77745900e+08j],
         [ 4.84763248e+08-1.03917874e+07j,
           1.00000000e+09+0.00000000e+00j,
           5.43869087e+08-5.97507660e+08j],
         [ 5.62495313e+08+5.77745900e+08j,
           5.43869087e+08+5.97507660e+08j,
           1.00000000e+09+0.00000000e+00j]],
 
        [[ 1.00000000e+09+0.00000000e+00j,
           3.12426589e+08+1.82528476e+06j,
           2.14669698e+08-6.77293097e+08j],
         [ 3.12426589e+08-1.82528476e+06j,
           1.00000000e+09+0.